In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)
    
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, norm: bool | None = True, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
        ]
        
        if norm:
            layers.insert(1, nn.BatchNorm2d(out_channels))
            layers.insert(4, nn.BatchNorm2d(out_channels))
        
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)
    

class ResizeProject(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.project = ConvBlock(in_channels, out_channels)

    def forward(self, x, target_size):
        if x.shape[-2:] != target_size:
            x = F.interpolate(x, size=target_size, mode="bilinear", align_corners=False)
        return self.project(x)

def init_weights_kaiming(module: nn.Module) -> None:
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


In [16]:
class UNET3P(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=2,
        base_ch=64,
        cat_ch=64,
        deep_supervision=True,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        fusion_ch = cat_ch * 5

        self.enc1 = DoubleConv(in_channels, base_ch)
        self.enc2 = DoubleConv(base_ch, base_ch * 2)
        self.enc3 = DoubleConv(base_ch * 2, base_ch * 4)
        self.enc4 = DoubleConv(base_ch * 4, base_ch * 8)
        self.enc5 = DoubleConv(base_ch * 8, base_ch * 16)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.proj_d4 = nn.ModuleDict(
            {
                "e1": ResizeProject(base_ch, cat_ch),
                "e2": ResizeProject(base_ch * 2, cat_ch),
                "e3": ResizeProject(base_ch * 4, cat_ch),
                "e4": ResizeProject(base_ch * 8, cat_ch),
                "e5": ResizeProject(base_ch * 16, cat_ch),
            }
        )
        self.proj_d3 = nn.ModuleDict(
            {
                "e1": ResizeProject(base_ch, cat_ch),
                "e2": ResizeProject(base_ch * 2, cat_ch),
                "e3": ResizeProject(base_ch * 4, cat_ch),
                "d4": ResizeProject(fusion_ch, cat_ch),
                "e5": ResizeProject(base_ch * 16, cat_ch),
            }
        )
        self.proj_d2 = nn.ModuleDict(
            {
                "e1": ResizeProject(base_ch, cat_ch),
                "e2": ResizeProject(base_ch * 2, cat_ch),
                "d3": ResizeProject(fusion_ch, cat_ch),
                "e4": ResizeProject(base_ch * 8, cat_ch),
                "e5": ResizeProject(base_ch * 16, cat_ch),
            }
        )
        self.proj_d1 = nn.ModuleDict(
            {
                "e1": ResizeProject(base_ch, cat_ch),
                "d2": ResizeProject(fusion_ch, cat_ch),
                "e3": ResizeProject(base_ch * 4, cat_ch),
                "e4": ResizeProject(base_ch * 8, cat_ch),
                "e5": ResizeProject(base_ch * 16, cat_ch),
            }
        )

        self.dec4 = ConvBlock(fusion_ch, fusion_ch)
        self.dec3 = ConvBlock(fusion_ch, fusion_ch)
        self.dec2 = ConvBlock(fusion_ch, fusion_ch)
        self.dec1 = ConvBlock(fusion_ch, fusion_ch)

        self.final_conv = ConvBlock(in_channels=fusion_ch, out_channels=out_channels)#nn.Conv2d(fusion_ch, out_channels, kernel_size=1)
        self.supervision = nn.ModuleList(
            [
                nn.Conv2d(base_ch * 16, out_channels, kernel_size=1),
                nn.Conv2d(fusion_ch, out_channels, kernel_size=1),
                nn.Conv2d(fusion_ch, out_channels, kernel_size=1),
                nn.Conv2d(fusion_ch, out_channels, kernel_size=1)
            ]
        )

    def _fuse(self, projections, features, target_size):
        return torch.cat([projections[name](features[name], target_size) for name in projections], dim=1)

    def forward(self, x):

        image_size = x.shape[-2:]

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        e5 = self.enc5(self.pool(e4))

        d4 = self.dec4(
            self._fuse(
                self.proj_d4,
                {"e1": e1, "e2": e2, "e3": e3, "e4": e4, "e5": e5},
                e4.shape[-2:],
            )
        )        
        d3 = self.dec3(
            self._fuse(
                self.proj_d3,
                {"e1": e1, "e2": e2, "e3": e3, "d4": d4, "e5": e5},
                e3.shape[-2:],
            )
        )
        d2 = self.dec2(
            self._fuse(
                self.proj_d2,
                {"e1": e1, "e2": e2, "d3": d3, "e4": e4, "e5": e5},
                e2.shape[-2:],
            )
        )
        d1 = self.dec1(
            self._fuse(
                self.proj_d1,
                {"e1": e1, "d2": d2, "e3": e3, "e4": e4, "e5": e5},
                e1.shape[-2:],
            )
        )
        output = self.final_conv(d1)

        print(d1.shape, d2.shape, d3.shape, d4.shape)


        if not self.deep_supervision:
            return output

        return [
            F.interpolate(self.supervision[0](e5), size=image_size, mode="bilinear", align_corners=False),
            F.interpolate(self.supervision[1](d4), size=image_size, mode="bilinear", align_corners=False),
            F.interpolate(self.supervision[2](d3), size=image_size, mode="bilinear", align_corners=False),
            F.interpolate(self.supervision[3](d2), size=image_size, mode="bilinear", align_corners=False),
            output
        ]


In [21]:
if __name__ == "__main__":
    model = UNET3P(in_channels=1, out_channels=2, deep_supervision=True)

    model.apply(init_weights_kaiming)
    
    device = "cpu"
    model = model.to(device)
    model.train()

In [22]:
print(f"{'Layer Name':<25} | {'Mean':<10} | {'Std Dev':<10} | {'Data Snapshot'}")
print("-" * 70)
for name, param in model.named_parameters():
    if 'weight' in name and ('supervision' in name or 'final' in name):
        mean = param.data.mean().item()
        std = param.data.std().item()
        # Take a look at the first 3 values just to be sure they aren't all zeros
        snapshot = param.data.flatten()[-3:].tolist()
        
        print(f"{name:<25} | {mean:10.4f} | {std:10.4f} | {snapshot}")

Layer Name                | Mean       | Std Dev    | Data Snapshot
----------------------------------------------------------------------
final_conv.block.0.weight |     0.0010 |     0.3354 | [0.4424378573894501, -0.026186054572463036, -0.048815082758665085]
final_conv.block.1.weight |     1.0000 |     0.0000 | [1.0, 1.0]
supervision.0.weight      |     0.0538 |     1.0042 | [0.2503502666950226, -1.081989049911499, -1.0958136320114136]
supervision.1.weight      |     0.0165 |     0.9784 | [-0.5197434425354004, -1.0154824256896973, 1.0127404928207397]
supervision.2.weight      |    -0.0467 |     0.9967 | [0.21703194081783295, -0.5097513794898987, -0.7353441715240479]
supervision.3.weight      |     0.0581 |     0.9752 | [0.6175125241279602, -0.9797919392585754, 1.0648635625839233]


In [11]:
if __name__ == "__main__":
    model = UNET3P(in_channels=1, out_channels=2, deep_supervision=True)

    device = "cpu"
    model = model.to(device)
    model.train()

    batch_size = 3
    in_channels = 1
    height = width = 128
    x = torch.randn(batch_size, in_channels, height, width, device=device)

    #print(model)
    out = model(x)

    #print("Output shapes:")
    #for i, o in enumerate(out):
    #    print(f"  Level {i}: {o.shape}")

torch.Size([3, 320, 128, 128]) torch.Size([3, 320, 64, 64]) torch.Size([3, 320, 32, 32]) torch.Size([3, 320, 16, 16])
